In [ ]:
import pandas as pd

## Demand for the energy transition for 2023

In [ ]:
df_iea_demand_nze = pd.read_excel(r'data/data_metals_clean.xlsx', sheet_name='EI_markets')

In [ ]:
df_iea_demand_nze = df_iea_demand_nze[df_iea_demand_nze['Scenario'] == 'NZE'] 

In [ ]:
df_iea_demand_nze.columns = df_iea_demand_nze.columns.map(str)

In [ ]:
df_iea_demand_nze = df_iea_demand_nze.rename(columns={'2022':'2023'})
#df_iea_demand_nze = df_iea_demand_nze.drop(columns=['2030', '2035', '2040', '2045', '2050'])

In [ ]:
df_iea_demand_nze

In [ ]:
df_iea_demand_nze = df_iea_demand_nze.groupby('Metal', as_index=False)[['2023', '2030', '2035', '2040', '2045', '2050']].sum()

In [ ]:
df_iea_demand_nze

In [ ]:
df_iea_demand_nze = pd.melt(
    df_iea_demand_nze, 
    id_vars=['Metal'], 
    var_name='Year', 
    value_name='Energy transition (kt)'
)

In [ ]:
# We drop Hafnium as there is no production data, and rename graphite
df_iea_demand_nze['Metal'] = df_iea_demand_nze['Metal'].replace("Graphite (battery-grade)", "Graphite") 
df_iea_demand_nze.drop(df_iea_demand_nze[df_iea_demand_nze['Metal'] == 'Hafnium'].index, inplace=True)

In [ ]:
df_iea_demand_nze

In [ ]:
df_iea_demand_nze.to_csv(r'df_iea_demand_nze.csv', index=False)

In [ ]:
df_iea_demand_nze_2023 = df_iea_demand_nze[df_iea_demand_nze['Year'] == '2023']

## Production data for 2023

In [ ]:
df_production = pd.read_excel(r'data/data_minerals/data_availability_marin.xlsx', sheet_name='for_pandas')

In [ ]:
col_to_keep = ['Metal', 'Total_refinery_production', 'Year', 'Unit']

In [ ]:
df_production = df_production[col_to_keep]
df_production['Year'] = df_production['Year'].astype(str)
df_production

In [ ]:
def convert_to_kt(row):
    if row['Unit'] == 't':
        return row['Total_refinery_production'] / 1_000
    elif row['Unit'] == 'kg':
        return row['Total_refinery_production'] / 1_000_000
    else:  # already in kt
        return row['Total_refinery_production']

In [ ]:
df_production['Total_refinery_production'] = df_production.apply(convert_to_kt, axis=1)
df_production.rename(columns={'Total_refinery_production': 'Total refinery production (kt)'}, inplace=True)
df_production.drop(columns=['Unit'], inplace=True)

In [ ]:
df_production

## Compute demand for the rest of the economy

In [ ]:
df = df_production.merge(df_iea_demand_nze_2023, on=['Metal', 'Year'], how='left')

In [ ]:
df

In [ ]:
df['Rest of the economy (kt)'] = df['Total refinery production (kt)'] - df['Energy transition (kt)']

In [ ]:
# Let's remove Terbium for now to avoid negative values
df = df[df['Metal'] != 'Terbium']

In [ ]:
# Move 'Year' to the end
cols = [col for col in df.columns if col != 'Year'] + ['Year']
df = df[cols]

In [ ]:
df.drop(columns=['Total refinery production (kt)'], inplace=True)

In [ ]:
df

In [ ]:
df.to_excel(r'df.xlsx')

## Project production to 2050

In [ ]:
import pandas as pd

def project_demand(df_base, df_iea, iam_df, parameter='gdp', scenario='SSP2'):
    """
    Project annual demand for energy transition and rest of the economy from base year (2023)
    to target years using IAM trends (GDP or population) for a given scenario.

    Args:
        df_base (pd.DataFrame): Base DataFrame with columns 
            ['Metal', 'Energy transition (kt)', 'Rest of the economy (kt)', 'Year'].
            Must include Year=2023 values.
        df_iea (pd.DataFrame): IEA demand DataFrame with ['Metal', 'Year', 'Energy transition (kt)']
            for years (e.g., 2023, 2030, …, 2050).
        iam_df (pd.DataFrame): IAM projections DataFrame with columns
            ['model','scenario','region','variable','unit', <year columns as strings>].
        parameter (str): 'gdp' or 'population' to choose trend variable.
        scenario (str): IAM scenario name, e.g., 'SSP2'.

    Returns:
        pd.DataFrame: Long-format DataFrame with columns
        ['Metal', 'Year', 'Energy transition (kt)', 'Rest of the economy (kt)'] for each Metal-Year.
    """
    # 1) Map parameter to IAM variable
    var_map = {'gdp': 'GDP|PPP', 'population': 'Population'}
    param_label = var_map.get(parameter.lower())
    if param_label is None:
        raise ValueError("parameter must be 'gdp' or 'population'")

    # 2) Filter IAM data
    iam_sub = iam_df[
        (iam_df['scenario'] == scenario) &
        (iam_df['region'] == 'World') &
        (iam_df['variable'] == param_label)
    ]
    if iam_sub.empty:
        raise ValueError(f"No IAM data for scenario={scenario}, variable={param_label}")

    # 3) Identify year columns (strings like '2025', etc.)
    year_cols = [c for c in iam_sub.columns if c.isdigit()]

    # 4) Melt to long form
    iam_long = iam_sub.melt(
        id_vars=['model','scenario','region','variable','unit'],
        value_vars=year_cols,
        var_name='Year',
        value_name='Value'
    )
    iam_long['Year'] = iam_long['Year'].astype(int)

    # 5) — NEW: aggregate across models so each Year is unique
    iam_yearly = (
        iam_long
        .groupby('Year', as_index=True)['Value']
        .mean()
        .sort_index()
    )

    # 6) Determine target years from IEA data
    df_iea = df_iea.copy()
    df_iea['Year'] = pd.to_numeric(df_iea['Year'], errors='coerce').astype(int)
    years = sorted(df_iea['Year'].dropna().unique())

    # 7) Interpolate IAM trend to include all target years + base year 2023
    interp_index = sorted(set(iam_yearly.index) | set(years) | {2023})
    iam_interp = iam_yearly.reindex(interp_index).interpolate(method='linear')

    # 8) Compute relative growth factors (base 2023 = 1.0)
    base_val = iam_interp.loc[2023]
    factors = {yr: iam_interp.loc[yr] / base_val for yr in years}

    # 9) Extract base rest-of-economy values for 2023
    df_base = df_base.copy()
    df_base['Year'] = pd.to_numeric(df_base['Year'], errors='coerce').astype(int)
    base_rest = df_base[df_base['Year'] == 2023].set_index('Metal')['Rest of the economy (kt)']

    # 10) Build the projected records
    records = []
    for metal, rest_2023 in base_rest.items():
        for yr in years:
            # IEA energy-transition demand
            et_vals = df_iea[
                (df_iea['Metal'] == metal) & (df_iea['Year'] == yr)
            ]['Energy transition (kt)']
            et_value = et_vals.iloc[0] if not et_vals.empty else pd.NA

            # Project rest-of-economy demand
            rest_proj = rest_2023 * factors[yr]

            records.append({
                'Metal': metal,
                'Year': yr,
                'Energy transition (kt)': et_value,
                'Rest of the economy (kt)': rest_proj
            })

    return pd.DataFrame.from_records(records)


In [ ]:
df_ssp = pd.read_csv(r'data/data_ssp/iamdf_ssp.csv')

In [ ]:
proj_df = project_demand(
    df_base=df, 
    df_iea=df_iea_demand_nze, 
    iam_df=df_ssp, 
    parameter='gdp', 
    scenario='SSP2'
)


In [ ]:
proj_df

In [ ]:
proj_df.to_csv(r'results/energy_transition_vs_roe.csv', index=False)